In [1]:
import pandas as pd

# Load raw data
df = pd.read_csv('../data/raw/Sample - Superstore.csv', encoding='windows-1252')

# Standardize column headers to snake_case
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

# Parse date strings to datetime
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Verify schema
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   row_id         9994 non-null   int64         
 1   order_id       9994 non-null   object        
 2   order_date     9994 non-null   datetime64[ns]
 3   ship_date      9994 non-null   datetime64[ns]
 4   ship_mode      9994 non-null   object        
 5   customer_id    9994 non-null   object        
 6   customer_name  9994 non-null   object        
 7   segment        9994 non-null   object        
 8   country        9994 non-null   object        
 9   city           9994 non-null   object        
 10  state          9994 non-null   object        
 11  postal_code    9994 non-null   int64         
 12  region         9994 non-null   object        
 13  product_id     9994 non-null   object        
 14  category       9994 non-null   object        
 15  sub_category   9994 n

In [3]:
# 1. Cleaned master transactional file
df.to_csv('../data/processed/cleaned_superstore.csv', index=False)

# 2. Customer Dimension
dim_customers = df[['customer_id', 'customer_name', 'segment']].drop_duplicates().reset_index(drop=True)
dim_customers.to_csv('../data/processed/dim_customers.csv', index=False)

# 3. Product Dimension
dim_products = df[['product_id', 'product_name', 'category', 'sub_category']].drop_duplicates().reset_index(drop=True)
dim_products.to_csv('../data/processed/dim_products.csv', index=False)

# 4. Geography Dimension
dim_geography = df[['country', 'city', 'state', 'postal_code', 'region']].drop_duplicates().reset_index(drop=True)
dim_geography['geo_id'] = dim_geography.index + 1
dim_geography.to_csv('../data/processed/dim_geography.csv', index=False)

print("Preprocessed files successfully created in data/processed/")

Preprocessed files successfully created in data/processed/


In [1]:
import os
import numpy as np
import pandas as pd

# 1. Load raw Superstore dataset
raw_data_path = '../data/raw/Sample - Superstore.csv'
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

df = pd.read_csv(raw_data_path, encoding='windows-1252')

# Standardize column headers to snake_case
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

# Parse date strings to proper date objects
df['order_date'] = pd.to_datetime(df['order_date']).dt.date
df['ship_date'] = pd.to_datetime(df['ship_date']).dt.date

# -------------------------------------------------------------
# 2. Extract and clean 'customers' entity
# -------------------------------------------------------------
df_customers = df[[
    'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region'
]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

# Rename to match database schema
df_customers.rename(columns={'customer_name': 'name'}, inplace=True)
df_customers.to_csv(f'{processed_dir}/customers.csv', index=False)

# -------------------------------------------------------------
# 3. Extract and clean 'products' entity
# -------------------------------------------------------------
# Calculate unit_price safely (sales / quantity)
df['calculated_unit_price'] = (df['sales'] / df['quantity']).round(2)

# Group to ensure unique product_id with consistent name and category
df_products = df.groupby('product_id').agg({
    'product_name': 'first',
    'category': 'first',
    'calculated_unit_price': 'median'
}).reset_index()

df_products.rename(columns={
    'id': 'product_id',
    'product_name': 'name',
    'calculated_unit_price': 'unit_price'
}, inplace=True)
df_products.to_csv(f'{processed_dir}/products.csv', index=False)

# -------------------------------------------------------------
# 4. Extract 'sales' (order transactions) entity
# -------------------------------------------------------------
df_sales = df[[
    'row_id', 'product_id', 'customer_id', 'quantity', 'sales', 'order_date'
]].copy()

df_sales.rename(columns={
    'row_id': 'id',
    'sales': 'sales_amount',
    'order_date': 'sale_date'
}, inplace=True)
df_sales.to_csv(f'{processed_dir}/sales.csv', index=False)

# -------------------------------------------------------------
# 5. Generate 'inventory' records from products
# -------------------------------------------------------------
np.random.seed(42)
df_inventory = pd.DataFrame({
    'id': range(1, len(df_products) + 1),
    'product_id': df_products['product_id'],
    'stock_level': np.random.randint(5, 150, size=len(df_products)),
    'reorder_point': np.random.choice([15, 20, 25, 30], size=len(df_products))
})
df_inventory.to_csv(f'{processed_dir}/inventory.csv', index=False)

# -------------------------------------------------------------
# 6. Generate 'invoices' records from sales
# -------------------------------------------------------------
df_invoices = pd.DataFrame({
    'id': range(1, len(df_sales) + 1),
    'sale_id': df_sales['id'],
    'amount': df_sales['sales_amount'],
    'payment_status': np.random.choice(['Paid', 'Pending'], size=len(df_sales), p=[0.92, 0.08])
})
df_invoices.to_csv(f'{processed_dir}/invoices.csv', index=False)

# -------------------------------------------------------------
# 7. Generate default 'users' table (RBAC roles)
# -------------------------------------------------------------
users_data = [
    {'id': 1, 'name': 'Owner Admin', 'email': 'owner@marketmind.ai', 'role': 'Business Owner'},
    {'id': 2, 'name': 'Downtown Manager', 'email': 'manager@marketmind.ai', 'role': 'Store Manager'},
    {'id': 3, 'name': 'Sales Lead', 'email': 'sales@marketmind.ai', 'role': 'Sales Executive'},
    {'id': 4, 'name': 'System Administrator', 'email': 'admin@marketmind.ai', 'role': 'Administrator'}
]
df_users = pd.DataFrame(users_data)
df_users.to_csv(f'{processed_dir}/users.csv', index=False)

print("Data preprocessing complete. Cleaned files saved to '../data/processed/'.")

Data preprocessing complete. Cleaned files saved to '../data/processed/'.
